<a href="https://colab.research.google.com/github/SiyumiJayawardhane/sinhala-wsd/blob/sinhala_textbook_extraction/textbook_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Textbook sentence extraction for Sinhala WSD

Extracts natural sentences containing the 65 target words from the EPD textbook
*සිංහල සාහිත්‍ය සංග්‍රහය* (Grade 10–11, 7th printing 2020, first printed 2014, ISBN 978-955-25-0381-8).

**Input:** book PDF (not redistributed, all rights reserved), `WSD_Polysemy_Words.xlsx` (sheet `target_words`).
**Output:** target-word sentences (CSV master with book + page, TXT for the annotation tool), metadata JSON, spot-check sheet.
**Run:** top to bottom in Google Colab. Random seed = 42.

1. Setup

*   Installs pymupdf (to read the PDF’s text) and pandukabhaya (to convert FM-Abhaya to Unicode), then connects Drive.

In [17]:
!pip install -q pymupdf pandukabhaya

In [18]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, json, random, subprocess, unicodedata, datetime
from collections import Counter
from pandukabhaya import Converter

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


2. Load the book
* Book details used for citation and provenance. `page_offset`
converts a PDF page number to the printed page number (printed page = PDF page − offset).



In [19]:
BASE    = "/content/drive/MyDrive/Research-WSD-New"
PDF_DIR = f"{BASE}/textbooks"
OUT_DIR = f"{BASE}/textbooks/Ext_Final"

BOOKS = [
    {
        "book_id":     "sinhala_lit_g10_11",
        "pdf":         "sinhala lit g-10-11.pdf",
        "title":       "සිංහල සාහිත්‍ය සංග්‍රහය (Grade 10–11)",
        "publisher":   "Educational Publications Department",
        "grade":       "10-11",
        "year":        "2020",
        "edition":     "7th printing (first printed 2014)",
        "isbn":        "978-955-25-0381-8",
        "url":         "http://edupub.gov.lk/Administrator/Sinhala/10/sin%20lit%20g-10-11/sinhala%20lit%20g-10-11.pdf",
        "page_offset": 10,
    },
]
for b in BOOKS:
    b["pdf_path"] = os.path.join(PDF_DIR, b["pdf"])
    b["out"] = os.path.join(OUT_DIR, b["book_id"]); os.makedirs(b["out"], exist_ok=True)
    print(("OK      " if os.path.exists(b["pdf_path"]) else "MISSING ") + b["pdf_path"])

OK      /content/drive/MyDrive/Research-WSD-New/textbooks/sinhala lit g-10-11.pdf


3. Read the text and check its encoding
*  Reads every text span with its font and x/y position. The book uses the legacy FM-Abhaya font, so it needs conversion; a scanned book (no text layer) would go to Document AI OCR instead.
pdftotext was rejected: it reads two-column poems column by column (losing rhyme words) and cannot see fonts (English text was "converted" into fake Sinhala).



In [20]:
# First option was pdftotext but (1) it reads two-column poems column by column, so a verse line loses its rhyme word
# (2) it can't see fonts, so English words (Times font) were "converted" into fake Sinhala.
# PyMuPDF gives each text span with its font and x/y position, so we can fix both.
import pymupdf
COL_GAP, WORD_GAP, LINE_TOL = 12.0, 1.5, 3.0

def read_spans(path):
    """List of pages; each page = list of spans {text, font, bbox}."""
    with pymupdf.open(path) as doc:
        return [[s for b in p.get_text("dict")["blocks"] for l in b.get("lines", []) for s in l["spans"]
                 if s["text"].strip()] for p in doc]

def detect_encoding(pages):
    """'fm' if most text is in FM (legacy) fonts, 'unicode' if it is already Sinhala Unicode,
    'none' if there is (almost) no text layer -> scanned -> needs Document AI OCR."""
    chars = Counter()
    for p in pages:
        for s in p:
            chars["fm" if s["font"].startswith("FM") else
                  "unicode" if SINHALA.search(s["text"]) else "other"] += len(s["text"])
    total = sum(chars.values())
    if total < 500:
        return "none", chars
    return ("fm" if chars["fm"] > chars["unicode"] else "unicode"), chars

SINHALA = re.compile(r"[\u0D80-\u0DFF]")
LATIN   = re.compile(r"[A-Za-z]")
for b in BOOKS:
    b["spans"] = read_spans(b["pdf_path"])
    b["encoding"], chars = detect_encoding(b["spans"])
    fonts = Counter(s["font"] for p in b["spans"] for s in p)
    print(f'{b["book_id"]}: {len(b["spans"])} pages -> {b["encoding"]} | chars by type {dict(chars)}')
    print("   fonts:", fonts.most_common(8))

sinhala_lit_g10_11: 140 pages -> fm | chars by type {'fm': 224027, 'other': 1912}
   fonts: [('FMAbhayax', 6420), ('TimesNewRomanPSMT', 1259), ('FMAbabldBold', 446), ('MinionPro-Regular', 144), ('FMEmaneex', 70), ('FMSamanthaBoldx', 16), ('TimesNewRomB', 4), ('TimesNewRomanPS-BoldMT', 3)]


4. Convert FM-Abhaya to Unicode
* Converts only FM-font text and keeps English (Times/Minion) text unchanged. Fixes glyphs the converter misses in this book, removes "fake bold" duplicate spans, rebuilds verse lines (rhyme words printed apart) and real two-column blocks, then NFC-normalises.



In [21]:
conv = Converter("fm_abhaya")
NOISE = re.compile(r"[\u00AD\u200B\uFEFF]")

# Glyphs the fm_abhaya table leaves unconverted in this book (checked on real words):
# `ඵ -> ළු (නෙළුම්, පුළුවන්), `ද -> ඳ (අඳින්නේ), `ඩ -> ඬ (පඬිතුමා), ¥ -> දූ (දූතයා, දූවිලි),
# i after a Sinhala letter -> ද්‍ය (විද්‍යාලය, පද්‍ය).
POST_FIXES = [(re.compile(r"`ඵ"), "ළු"), (re.compile(r"`ද"), "ඳ"), (re.compile(r"`ඩ"), "ඬ"),
              (re.compile(r"¥"), "දූ"), (re.compile(r"(?<=[\u0D80-\u0DFF])i"), "ද්\u200Dය")]

def fm_to_unicode(t):
    # “ and ‘ are used as quote marks in this book, but the table maps “ -> ර්‍ණ and ' ‘' -> ි.
    # Protect them BEFORE converting so real letters are untouched.
    t = t.replace("“", "\u0001").replace("‘", "\u0002")
    t = conv.convert(t).replace("\u0001", "“").replace("\u0002", "‘")
    for pat, rep in POST_FIXES:
        t = pat.sub(rep, t)
    return t

def segment_text(spans, encoding):
    """Text of spans that sit close together on one line. FM-font runs are converted together;
    other fonts (Times, Minion = English, ISBN) are kept exactly as they are."""
    out, run, run_fm, prev_x1 = [], "", None, None
    for s in spans:
        fm = encoding == "fm" and s["font"].startswith("FM")
        txt = s["text"]
        if prev_x1 is not None and s["bbox"][0] - prev_x1 > WORD_GAP and not txt.startswith(" "):
            txt = " " + txt
        if run_fm is None or fm == run_fm:
            run += txt; run_fm = fm
        else:
            out.append(fm_to_unicode(run) if run_fm else run); run, run_fm = txt, fm
        prev_x1 = s["bbox"][2]
    if run:
        out.append(fm_to_unicode(run) if run_fm else run)
    return re.sub(r"\s+", " ", "".join(out)).strip()

DEP_SIGN = re.compile(r"^[\u0DCA-\u0DDF\u0D82\u0D83]")
NUM_ONLY = re.compile(r"^\(?\d{1,3}[.)]?$")

GLOSS_ROW = re.compile(r"^\S[^.?!]{0,60}\s-\s")
is_short = lambda t: all(len(w) <= 2 for w in t.split())

def merge_segments(segs):
    """Pieces of one printed line (left to right, already without a second column) -> (lines, new_unit).
    - rhyme word / syllables printed apart on the right of a verse line -> joined to the line
      (ගොදු + රු = ගොදුරු, පඳුරු + ත ළා = පඳුරු තළා)
    - glossary 'word   - meaning' -> one row; two glossary entries on one printed line -> two rows
    - label 'අන්වය :   text' -> one line
    A stanza / question number at the start ("28.") or a glossary row starts a new unit."""
    new_unit = False
    if len(segs) > 1 and NUM_ONLY.match(segs[0]):
        segs, new_unit = segs[1:], True
    out, cur = [], segs[0]
    for i, nxt in enumerate(segs[1:], start=1):
        next_is_meaning = i + 1 < len(segs) and segs[i + 1].startswith("-")
        if " - " in cur and next_is_meaning:
            out.append(cur); cur = nxt
        elif DEP_SIGN.match(nxt) or (is_short(nxt) and len(cur.split()[-1]) <= 1):
            cur += nxt.replace(" ", "")
        elif is_short(nxt):
            cur += " " + nxt.replace(" ", "")
        else:
            cur += " " + nxt
    out.append(cur)
    return out, new_unit or bool(GLOSS_ROW.match(out[0]))

def rebuild_page(spans, encoding):
    """Group spans into printed lines (same baseline), split lines at big gaps, merge, and return the
    page text: left column first, then the right column (only where a page really has two)."""
    spans = sorted(spans, key=lambda s: (round(s["bbox"][3]), s["bbox"][0]))
    kept = []                                    # drop "fake bold" duplicates: same text printed twice
    for s in spans:                              # a few points apart (shadow effect)
        if not any(k["text"].replace(" ", "") == s["text"].replace(" ", "") and abs(k["bbox"][0] - s["bbox"][0]) < 5
                   and abs(k["bbox"][3] - s["bbox"][3]) < 4 for k in kept[-6:]):
            kept.append(s)
    spans, rows = kept, []
    for s in spans:
        if rows and abs(rows[-1][0] - s["bbox"][3]) <= LINE_TOL:
            rows[-1][1].append(s)
        else:
            rows.append([s["bbox"][3], [s]])
    # pass 1: cut every printed line into pieces at big gaps, remembering where each piece starts (x)
    lines = []
    for _, ss in rows:
        ss.sort(key=lambda s: s["bbox"][0])
        groups, cur = [], [ss[0]]
        for s in ss[1:]:
            if s["bbox"][0] - cur[-1]["bbox"][2] > COL_GAP:
                groups.append(cur); cur = [s]
            else:
                cur.append(s)
        groups.append(cur)
        segs = [(segment_text(g, encoding), g[0]["bbox"][0]) for g in groups]
        lines.append([(t, x) for t, x in segs if t])
    # pass 2: find a real two-column block (e.g. a two-column song). A "right-column start" is a piece
    # that begins right of 45% of the page width and is several real words — not a rhyme word, not
    # spaced rhyme syllables, not a glossary meaning. If a page has 4+ such lines, every piece that
    # starts at or right of that column, between the first and last such line, goes to column 2.
    width = max((s["bbox"][2] for s in spans), default=0)
    def col2_start(t, x):
        return x > 0.45 * width and " " in t and not is_short(t) and not t.startswith("-")
    hits = [i for i, segs in enumerate(lines)
            if any(col2_start(t, x) for t, x in segs) and not any(t.startswith("-") for t, _ in segs)]
    if len(hits) >= 4:
        boundary = min(x for i in hits for t, x in lines[i] if col2_start(t, x)) - 10
        lo, hi = hits[0], hits[-1]
    else:
        boundary, lo, hi = width + 1, -1, -1
    streams = {}
    for i, segs in enumerate(lines):
        if not segs:
            continue
        in_block = lo <= i <= hi
        to_col2 = lambda t, x: in_block and x >= boundary and not is_short(t)   # rhyme syllables stay
        left = [t for t, x in segs if not to_col2(t, x)]
        rcol = " ".join(t for t, x in segs if to_col2(t, x))
        rows, new_unit = merge_segments(left) if left else ([], False)
        targets = [(0, t) for t in rows] + ([(1, rcol)] if rcol else [])
        for col, text in targets:
            st = streams.setdefault(col, [])
            if new_unit or (col == 0 and GLOSS_ROW.match(text)):
                st.append("")                      # blank line = hard boundary before this line
            st.append(text)
            if col == 0 and GLOSS_ROW.match(text):
                st.append("")
    page = "\n\n".join("\n".join(streams[k]) for k in sorted(streams))
    return NOISE.sub("", unicodedata.normalize("NFC", page))

for b in BOOKS:
    if b["encoding"] == "none":
        print(f'{b["book_id"]}: no text layer — skip here, use Document AI OCR'); continue
    b["pages"] = [rebuild_page(p, b["encoding"]) for p in b["spans"]]
    json.dump(b["pages"], open(f'{b["out"]}/{b["book_id"]}_pages_unicode.json', "w", encoding="utf-8"), ensure_ascii=False)
    print(f'{b["book_id"]}: {len(b["pages"])} pages converted')
    print("   sample:", re.sub(r"\s+", " ", b["pages"][min(40, len(b["pages"])-1)])[:200])

sinhala_lit_g10_11: 140 pages converted
   sample: (2) වනපෙතිනැවිදින් : වනපෙතින් + ඇවිදින් = වනපෙතිනැවිදින් (3) සුමුවෝ : සමූහ ව (4) නදළ මුවෝ : නද + අළ මුවෝ = නදළ මුවෝ අළ යනු අර (ඇරීමේ) ධාතුවේ අතීත කෘදන්තයි. නාදයට විස්මිත ව කට ඇර ගෙන සිටියා වූ අළ මුව ය


5. Check the conversion page by page
* After a good conversion almost no Latin letters remain. Pages with more than 2% Latin letters are flagged for manual review.

In [22]:
# After a good conversion almost no Latin letters remain. Leftover Latin = glyphs the converter
# did not map (or real English text on that page) have to look at those pages before using them.
LATIN_LEFT_MAX = 0.02          # flag a page if > 2% of its letters are still Latin

rows = []
for b in BOOKS:
    if "pages" not in b: continue
    leftovers = Counter()
    for i, p in enumerate(b["pages"]):
        s, l = len(SINHALA.findall(p)), len(LATIN.findall(p))
        leftovers.update(ch for ch in p if not (SINHALA.match(ch) or ch.isspace() or ch.isdigit()
                                               or ch in ".,;:!?()[]-–—'\"“”‘’/\u200C\u200D"))
        rows.append({"book": b["book_id"], "pdf_page": i + 1, "sinhala": s, "latin": l,
                     "flag": (s + l) > 0 and l / (s + l) > LATIN_LEFT_MAX})
    flagged = [r["pdf_page"] for r in rows if r["book"] == b["book_id"] and r["flag"]]
    print(f'{b["book_id"]}: {len(flagged)} flagged pages {flagged[:20]}')
    print("   most common leftover characters:", leftovers.most_common(15))

import pandas as pd
pd.DataFrame(rows).to_csv(f"{OUT_DIR}/conversion_page_check.csv", index=False, encoding="utf-8")

sinhala_lit_g10_11: 6 flagged pages [2, 4, 5, 8, 53, 54]
   most common leftover characters: [('i', 35), ('+', 30), ('e', 28), ('n', 24), ('a', 23), ('l', 17), ('t', 17), ('h', 12), ('=', 12), ('u', 11), ('o', 11), ('r', 11), ('g', 10), ('s', 8), ('d', 7)]


6. page exclusions
* Front matter (all PDF pages before printed page 1) and pages that failed the conversion check after manual review are excluded from the dataset.

In [23]:
# Pages to leave out of the dataset:
#  - front matter: every PDF page before printed page 1 (covers, preface, contents)
#  - pages you checked by eye and found badly converted (set after looking at flagged pages if no flagged pages MANUAL_EXCLUDE = set())
MANUAL_EXCLUDE = {53, 54}
for b in BOOKS:
    b["exclude_pages"] = set(range(1, b["page_offset"] + 1)) | MANUAL_EXCLUDE
    print(b["book_id"], "excluded PDF pages:", sorted(b["exclude_pages"]))

sinhala_lit_g10_11 excluded PDF pages: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 53, 54]


7. Split into sentences
- Page-number lines are dropped; prose is split at . ? ! ; long verse chunks are cut into 2-line units.
- Each unit is tagged:
  - `text`: the literature itself (prose, verse, drama dialogue) → used for annotation
  - `glossary`: "word - meaning" / "word : meaning" rows → excluded (definitions, not usage; would leak the sense)
  - `study_note`: අන්වය / අර්ථය / විස්තර explanations → excluded (paraphrase of the verse)
  - `grammar_note`: sandhi breakdowns such as "සර + අග + ඒ = සරාගේ" → excluded
- Drama dialogue ("සිංහබාහු : <line>") keeps the line and drops the speaker name.
- Units with fewer than 4 Sinhala letters are dropped.

In [24]:
VERSE_LINES = 2
PAGE_NUM = re.compile(r"[\divxlcIVXLC\s]+")
SPLIT    = re.compile(r"(?<=[.?!])\s+")
LABEL    = re.compile(r"\s*(?=(?:අන්වය|අර්ථය|විස්තර)\s*\*?\s*:)")
TOKEN_SI = re.compile(r"[\u0D80-\u0DFF\u200C\u200D]+")

GLOSS_DASH  = re.compile(r"^\s*(?:\(\d+\)\s*)?\S[^.?!]{0,60}\s[-–]\s")    # 'දිය - ලෝකයෙහි'
GLOSS_COLON = re.compile(r"^\s*(?:\(\d+\)\s*)?[^.?!:]{1,40}\s?:\s")      # '(2) සුමුවෝ : සමූහ ව'
NOTE        = re.compile(r"^\s*(?:\(\d+\)\s*)?(?:අන්වය|අර්ථය|විස්තර)\s*\*?\s*:")
SANDHI      = re.compile(r"\+.*=")                                       # 'සර + අග + ඒ = සරාගේ'
SPEAKER     = re.compile(r"^\s*([^\s:]{2,25})\s?:\s(.+)$")                # 'සිංහබාහු : <line>'
DIALOGUE_MIN_TOKENS = 6          # a line this long after one name = dialogue, not a glossary meaning

def unit_type_and_text(t):
    """-> (type, text). Dialogue keeps the line without the speaker name."""
    if NOTE.match(t):
        return "study_note", t
    if SANDHI.search(t):
        return "grammar_note", t
    m = SPEAKER.match(t)
    if m and len(TOKEN_SI.findall(m.group(2))) >= DIALOGUE_MIN_TOKENS:
        return "text", m.group(2).strip()
    if GLOSS_DASH.match(t) or GLOSS_COLON.match(t):
        return "glossary", t
    return "text", t

def split_units(pages, offset, exclude):
    units = []
    for pi, p in enumerate(pages):
        if pi + 1 in exclude:
            continue
        lines = [re.sub(r"[ \t]+", " ", l).strip() for l in p.split("\n")]
        lines = [l for l in lines if not PAGE_NUM.fullmatch(l)]
        text = LABEL.sub("\n\n", "\n".join(lines))
        text = re.sub(r"\n{3,}", "\n\n", text)
        for block in text.split("\n\n"):
            for s in SPLIT.split(block):
                s = s.strip()
                if not s:
                    continue
                if len(s) > 280 and s.count("\n") >= 2:
                    ls = s.split("\n")
                    chunks = [" ".join(ls[i:i + VERSE_LINES]) for i in range(0, len(ls), VERSE_LINES)]
                else:
                    chunks = [s.replace("\n", " ")]
                for c in chunks:
                    c = re.sub(r"\s+", " ", c).strip()
                    if len(SINHALA.findall(c)) < 4:
                        continue
                    typ, txt = unit_type_and_text(c)
                    bp = pi + 1 - offset
                    units.append({"sid": len(units), "pdf_page": pi + 1,
                                  "book_page": bp if bp >= 1 else "",
                                  "type": typ, "text": txt})
    return units

for b in BOOKS:
    if "pages" not in b:
        continue
    b["units"] = split_units(b["pages"], b["page_offset"], b["exclude_pages"])
    with open(f'{b["out"]}/{b["book_id"]}_sentences.jsonl', "w", encoding="utf-8") as f:
        for u in b["units"]:
            f.write(json.dumps({"book_id": b["book_id"], **u}, ensure_ascii=False) + "\n")
    with open(f'{b["out"]}/{b["book_id"]}_corpus.txt', "w", encoding="utf-8") as f:
        f.write("\n".join(u["text"] for u in b["units"]))
    print(f'{b["book_id"]}: {len(b["units"]):,} units |', dict(Counter(u["type"] for u in b["units"])))

sinhala_lit_g10_11: 3,082 units | {'text': 2532, 'glossary': 473, 'study_note': 69, 'grammar_note': 8}


In [25]:
# Read these by eye: glossary rows must be definitions, text rows must be real sentences/verse/dialogue.
# If real text appears under "glossary", tighten GLOSS_COLON or lower DIALOGUE_MIN_TOKENS and rerun cell 7.
for b in BOOKS:
    if "units" not in b:
        continue
    for typ in ["glossary", "study_note", "grammar_note", "text"]:
        pool = [u for u in b["units"] if u["type"] == typ]
        print(f"\n=== {typ} ({len(pool)}) ===")
        for u in random.Random(1).sample(pool, min(15, len(pool))):
            print(f'  p{u["pdf_page"]}: {u["text"][:120]}')


=== glossary (473) ===
  p28: සුප්‍රතිෂ්ඨිත - ස්ථිරව/ නො සැලෙන සේ පිහිටි
  p71: හේලි දහමක - සනාතන දහමක්
  p103: වයි - වෙයින්, වේගයෙන්, වහා
  p100: මදාරා පරසතු - සුර ලොව ඇතැයි කියන මල් වර්ග දෙකකි.
  p96: පොකුරු මී - පුෂ්පකර මදු, පියුම් පැණි
  p27: අන්‍යතර - එක්තරා
  p45: දෙපිට දෙපිට - දෙපැත්තක
  p27: ආශීර්විෂයක්හු - විෂ සහිත සර්පයකු
  p51: ගොඩැල්ල - උස් ගොඩබිම් කඩ
  p96: විලිකුන් - ඉදුණු රඹකැන් - කෙසෙල් කැන්
  p47: උරතුර - ළපැත්ත/ ළමැද
  p47: මන් - සිත්
  p87: ප්‍රතිමුක්ත - බැඳී, පැළඳී, එතී
  p46: සෙද සෙද - වහා
  p98: වැලි අතුළ මළු තැනි - වැලි අතුරන ලද (විහාර) මිදුල්

=== study_note (69) ===
  p33: අර්ථය : මහා ශිල්පී ගුත්තිල සක් දෙවිගේ වචනයෙන් සන්තෘප්ත ව, ‘මෙයින් දිනිමි’ යි ආඩම්බරපයට පත්ව පළමු කොට බමර තත සිඳ දැමී ය.
  p31: අන්වය : පහසු ය.
  p36: අන්වය : කන්කලු වෙණ නද ඇසුමෙහි ලොබිනේ, රන් රසු දැල් කොලහල වන බියෙනේ, සන්සල නො ව එහි පිළිරූ ලෙසිනේ නන් හය සෙන් සිටි මතිනේ 
  p32: විස්තර : තරගයේ ඉතා ම උග්‍ර අවස්ථාවේ දී සුර රජ අහසෙහි සිට, බෝසතුන්ට පමණක් ඇසෙන පරිද්දෙන් “තතක් සිඳුව” යි කීවේ ය.
  p4

8. Spot-check sheet
* 100 random `text` units (seed 42) are written to Excel and checked by hand (ok y/n, error type). The error rate is reported in the thesis.

In [26]:
SPOT_N, SEED = 100, 42
for b in BOOKS:
    if "units" not in b: continue
    pool = [u for u in b["units"] if u["type"] == "text"]
    sample = random.Random(SEED).sample(pool, min(SPOT_N, len(pool)))
    pd.DataFrame([{"sid": u["sid"], "pdf_page": u["pdf_page"], "book_page": u["book_page"],
                   "text": u["text"], "ok (y/n)": "", "error type": "", "note": ""}
                  for u in sorted(sample, key=lambda u: u["sid"])]
                 ).to_excel(f'{b["out"]}/{b["book_id"]}_spot_check.xlsx', index=False)
    print(f'{b["book_id"]}: {len(sample)} units -> {b["book_id"]}_spot_check.xlsx')

sinhala_lit_g10_11: 100 units -> sinhala_lit_g10_11_spot_check.xlsx


9. Filter records
* Records the source, method, tool versions, excluded pages and unit counts. Section 10 adds the extraction results to the same file.

In [27]:
import importlib.metadata as md
for b in BOOKS:
    if "units" not in b: continue
    meta = {k: b[k] for k in ["book_id", "pdf", "title", "publisher", "grade", "year", "url", "page_offset", "encoding"]}
    meta.update({
        "excluded_pdf_pages": sorted(b["exclude_pages"]),
        "unit_types": dict(Counter(u["type"] for u in b["units"])),
        "units_used_for_extraction": "type == 'text' only (glossary rows and study notes excluded)",
        "method": "PyMuPDF text spans (font + position); FM-font spans converted with pandukabhaya (fm_abhaya), "
                  "English (Times/Minion) spans kept; lines rebuilt by position (two-column poems, rhyme words); NFC"
                  if b["encoding"] == "fm" else "PyMuPDF text spans (already Unicode); lines rebuilt by position; NFC",
        "quote_fix": "FM glyphs “ and ‘ kept as quote marks before conversion" if b["encoding"] == "fm" else "",
        "pymupdf": md.version("pymupdf"),
        "pandukabhaya": md.version("pandukabhaya"),
        "pages": len(b["pages"]), "sentence_units": len(b["units"]),
        "date": datetime.date.today().isoformat(),
    })
    json.dump(meta, open(f'{b["out"]}/{b["book_id"]}_meta.json', "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print(json.dumps(meta, ensure_ascii=False, indent=1))

{
 "book_id": "sinhala_lit_g10_11",
 "pdf": "sinhala lit g-10-11.pdf",
 "title": "සිංහල සාහිත්‍ය සංග්‍රහය (Grade 10–11)",
 "publisher": "Educational Publications Department",
 "grade": "10-11",
 "year": "2020",
 "url": "http://edupub.gov.lk/Administrator/Sinhala/10/sin%20lit%20g-10-11/sinhala%20lit%20g-10-11.pdf",
 "page_offset": 10,
 "encoding": "fm",
 "excluded_pdf_pages": [
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  53,
  54
 ],
 "unit_types": {
  "text": 2532,
  "glossary": 473,
  "study_note": 69,
  "grammar_note": 8
 },
 "units_used_for_extraction": "type == 'text' only (glossary rows and study notes excluded)",
 "method": "PyMuPDF text spans (font + position); FM-font spans converted with pandukabhaya (fm_abhaya), English (Times/Minion) spans kept; lines rebuilt by position (two-column poems, rhyme words); NFC",
 "quote_fix": "FM glyphs “ and ‘ kept as quote marks before conversion",
 "pymupdf": "2.0",
 "pandukabhaya": "1.0.1",
 "pages": 140,
 "sentence_units": 3082,
 

10. Sentence extraction with base word
- Target words: 65 words from `WSD_Polysemy_Words.xlsx`. A cell with several spellings (e.g. සිළ / සිළි / සිළු) is ONE word; a unit containing any of them is one sample.
- Only `text` units are used. Exact duplicates and units with fewer than `MIN_TOKENS` Sinhala tokens are removed.
- Matching is on whole tokens, base form only, ZWJ/ZWNJ-insensitive, NFC.
- One row per (word, unit), even if the word appears twice.
- `GRAMMATICAL` (optional) lists checked patterns where the target is only a function word; every pattern is recorded in the metadata.
- Outputs: CSV master (book + page for every sentence), TXT in annotation-tool format, per-word summary.

In [28]:
import openpyxl
import pandas as pd

WORDS_XLSX = f"{BASE}/WSD_Polysemy_Words.xlsx"
WORDS_SHEET = "target_words"
EXPECTED_N = 65
MIN_TOKENS = 4          # units shorter than this give annotators no context

def load_words(path, sheet=WORDS_SHEET):
    """-> [(No, word cell, [spellings])]. 'සිළ / සිළි / සිළු' = ONE word with 3 spellings."""
    rows = openpyxl.load_workbook(path)[sheet].iter_rows(values_only=True)
    header = [str(h).strip() if h else "" for h in next(rows)]
    i_no, i_word = header.index("No"), header.index("Word")
    out, seen_w, seen_no = [], set(), set()
    for r in rows:
        r = tuple(r) + (None,) * len(header)
        if not r[i_word]:
            continue
        word = str(r[i_word]).strip()
        no = int(float(r[i_no]))
        spellings = [s.strip() for s in re.split(r"[/,]", word) if s.strip()]
        if word in seen_w or no in seen_no:
            raise ValueError(f"Duplicate word or number in sheet: {no} {word}")
        seen_w.add(word); seen_no.add(no)
        out.append((no, word, spellings))
    return out

WORDS = load_words(WORDS_XLSX)
assert len(WORDS) == EXPECTED_N, f"Expected {EXPECTED_N} words, got {len(WORDS)}"

JOINERS = re.compile(r"[\u200C\u200D]")
TOKEN   = re.compile(r"[\u0D80-\u0DFF\u200C\u200D]+")
key     = lambda t: JOINERS.sub("", unicodedata.normalize("NFC", t))   # ZWJ-insensitive

SPELLING2WORD = {}
for no, word, spellings in WORDS:
    for s in spellings:
        if key(s) in SPELLING2WORD and SPELLING2WORD[key(s)] != word:
            raise ValueError(f"Spelling {s} belongs to two words")
        SPELLING2WORD[key(s)] = word
word_no = {w: no for no, w, _ in WORDS}
print(f"{len(WORDS)} words | {len(SPELLING2WORD)} spellings")

# Optional: patterns where the target is only a function word. Add only after checking real sentences.
GRAMMATICAL = {
    # "කර": [r"කර\s+(?:ඇත|තිබේ|තිබුණි|ගැනීම|ගත්|දී|සිටි)"],
}
def only_grammatical(word, text, n_hits):
    pats = GRAMMATICAL.get(word, [])
    return bool(pats) and sum(len(re.findall(p, text)) for p in pats) >= n_hits

for b in BOOKS:
    if "units" not in b:
        continue
    stats = Counter()
    seen_text, rows = set(), []
    for u in b["units"]:
        if u["type"] != "text":
            stats[f'skipped_{u["type"]}'] += 1; continue
        k = key(re.sub(r"\s+", " ", u["text"]))
        if k in seen_text:
            stats["skipped_duplicate"] += 1; continue
        seen_text.add(k)
        toks = TOKEN.findall(u["text"])
        if len(toks) < MIN_TOKENS:
            stats["skipped_short"] += 1; continue
        stats["text_units_used"] += 1
        found, first = Counter(), {}
        for tok in toks:
            w = SPELLING2WORD.get(key(tok))
            if w:
                found[w] += 1; first.setdefault(w, tok)
        for w, n in found.items():                        # one row per (word, unit)
            if only_grammatical(w, u["text"], n):
                stats["skipped_grammatical"] += 1; continue
            rows.append({"No": word_no[w], "word": w, "matched": first[w], "book_id": b["book_id"],
                         "sid": u["sid"], "pdf_page": u["pdf_page"], "book_page": u["book_page"],
                         "sentence": u["text"]})

    cand = pd.DataFrame(rows, columns=["No", "word", "matched", "book_id", "sid",
                                       "pdf_page", "book_page", "sentence"]).sort_values(["No", "sid"])

    # MASTER file: keeps book + page for every sentence
    cand.to_csv(f'{b["out"]}/{b["book_id"]}_target_word_sentences.csv', index=False, encoding="utf-8")

    # Annotation-tool input. ';' separates sentences, so ';' inside a sentence -> ','
    with open(f'{b["out"]}/{b["book_id"]}_target_word_sentences.txt', "w", encoding="utf-8") as f:
        f.write("\n\n".join(f"s_word: {w}\nsentences: "
                            + "; ".join(s.replace(";", ",") for s in cand.loc[cand.word == w, "sentence"])
                            for _, w, _ in WORDS))

    summary = pd.DataFrame([{"No": no, "word": w, "spellings": " / ".join(sp),
                             "sentences": int((cand.word == w).sum()),
                             "by_spelling": ", ".join(f"{s} ({c})" for s, c in
                                                      cand.loc[cand.word == w, "matched"].value_counts().items())}
                            for no, w, sp in WORDS])
    summary.to_excel(f'{b["out"]}/{b["book_id"]}_target_word_summary.xlsx', index=False)

    # Add extraction results to the metadata written in section 9
    meta_path = f'{b["out"]}/{b["book_id"]}_meta.json'
    meta = json.load(open(meta_path, encoding="utf-8"))
    meta.update({
        "word_list": os.path.basename(WORDS_XLSX), "word_sheet": WORDS_SHEET, "n_words": len(WORDS),
        "min_tokens": MIN_TOKENS,
        "grammatical_patterns": GRAMMATICAL,
        "extraction_filters": dict(stats),
        "word_sentence_rows": len(cand),
        "distinct_units_with_target": int(cand.sid.nunique()),
        "words_with_0_sentences": list(summary.loc[summary.sentences == 0, "word"]),
    })
    json.dump(meta, open(meta_path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)

    print(f'{b["book_id"]}: {len(cand):,} (word, sentence) rows | {cand.sid.nunique():,} distinct units')
    print("   filters:", dict(stats))
    print("   words with 0 sentences:", list(summary.loc[summary.sentences == 0, "word"]))

65 words | 78 spellings
sinhala_lit_g10_11: 730 (word, sentence) rows | 564 distinct units
   filters: {'text_units_used': 2393, 'skipped_short': 122, 'skipped_glossary': 473, 'skipped_duplicate': 17, 'skipped_study_note': 69, 'skipped_grammar_note': 8}
   words with 0 sentences: ['වණ', 'රැස', 'කලාපය', 'නිදන', 'සමර', 'දීප', 'දල']
